In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# STAGE 1: ENVIRONMENT AND DATASET VERIFICATION
# LLAMA 3.2 3B QLORA FINE-TUNING
# ============================================================

import os
import glob
import sys
import torch
import pandas as pd
import importlib.metadata as metadata

print("=" * 65)
print("LLAMA 3.2 3B — ENVIRONMENT VERIFICATION")
print("=" * 65)

# 1. Python and PyTorch
print("\n[1] SOFTWARE ENVIRONMENT")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

# 2. GPU verification
print("\n[2] GPU VERIFICATION")

if torch.cuda.is_available():

    gpu_count = torch.cuda.device_count()

    print("Number of GPUs:", gpu_count)

    for i in range(gpu_count):
        print(f"\nGPU {i}:")
        print("Name:", torch.cuda.get_device_name(i))

        props = torch.cuda.get_device_properties(i)

        print(
            "VRAM:",
            round(props.total_memory / 1024**3, 2),
            "GB"
        )

else:
    print("WARNING: No CUDA GPU detected.")

# 3. Required libraries
print("\n[3] INSTALLED LIBRARIES")

required_packages = [
    "transformers",
    "peft",
    "bitsandbytes",
    "accelerate",
    "datasets",
    "trl",
    "scikit-learn",
    "pandas",
    "sentencepiece",
    "safetensors"
]

for package in required_packages:

    try:
        version = metadata.version(package)
        print(f"{package}: {version}")

    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# 4. Locate dataset
print("\n[4] DATASET DISCOVERY")

matches = glob.glob(
    "/kaggle/input/**/malware_dataset_*.csv",
    recursive=True
)

if not matches:
    raise FileNotFoundError(
        "malware_dataset_.csv was not found under /kaggle/input"
    )

DATASET_PATH = matches[0]

print("Dataset found:")
print(DATASET_PATH)

# 5. Load dataset
print("\n[5] DATASET LOADING")

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

# 6. Verify schema
print("\n[6] SCHEMA VERIFICATION")

assert "SHA256" in df.columns, "SHA256 column missing"
assert "Type" in df.columns, "Type column missing"

feature_columns = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

print("SHA256 column: Present")
print("Type column: Present")
print("PE feature count:", len(feature_columns))

# 7. Missing and duplicate values
print("\n[7] DATA QUALITY")

print("Missing values:", df.isnull().sum().sum())

print("Duplicate SHA256:", df["SHA256"].duplicated().sum())

print("\nClass distribution:")
display(
    df["Type"]
    .value_counts()
    .sort_index()
    .to_frame("Samples")
)

print("\n" + "=" * 65)
print("STAGE 1 VERIFICATION COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# STAGE 2: INSTALL MISSING DEPENDENCIES
# ============================================================

%pip install -q bitsandbytes==0.50.2 trl==1.18.0

import torch
import bitsandbytes
import trl
import transformers
import peft
import accelerate

print("=" * 60)
print("DEPENDENCY VERIFICATION")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

print("\nCUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2

print("\nSTAGE 2 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# STAGE 3: HUGGING FACE + GOOGLE DRIVE AUTHENTICATION
# ============================================================

import os
import base64
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

print("=" * 65)
print("STAGE 3: AUTHENTICATION")
print("=" * 65)

secrets = UserSecretsClient()

# ------------------------------------------------------------
# 1. HUGGING FACE AUTHENTICATION
# ------------------------------------------------------------

print("\n[1] HUGGING FACE")

HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN, add_to_git_credential=False)

user_info = whoami(token=HF_TOKEN)

print("Hugging Face authentication successful.")
print("Authenticated account:", user_info["name"])

# ------------------------------------------------------------
# 2. RCLONE CONFIGURATION
# ------------------------------------------------------------

print("\n[2] GOOGLE DRIVE CONFIGURATION")

RCLONE_CONFIG_B64 = secrets.get_secret("RCLONE_CONFIG_B64")

config_bytes = base64.b64decode(RCLONE_CONFIG_B64)

rclone_config_dir = Path.home() / ".config" / "rclone"
rclone_config_dir.mkdir(parents=True, exist_ok=True)

rclone_config_path = rclone_config_dir / "rclone.conf"

rclone_config_path.write_bytes(config_bytes)
os.chmod(rclone_config_path, 0o600)

print("rclone configuration restored.")

# ------------------------------------------------------------
# 3. VERIFY RCLONE
# ------------------------------------------------------------

result = subprocess.run(
    ["rclone", "listremotes"],
    capture_output=True,
    text=True
)

print("\nAvailable remotes:")
print(result.stdout.strip())

if result.returncode != 0:
    raise RuntimeError(result.stderr)

assert "gdrive:" in result.stdout, "Google Drive remote not found."

# ------------------------------------------------------------
# 4. CREATE BACKUP DIRECTORIES
# ------------------------------------------------------------

DRIVE_ROOT = "gdrive:llama32-malware-backups/llama32-training"

DRIVE_CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/data"

for folder in [DRIVE_CHECKPOINT_DIR, DRIVE_DATA_DIR]:

    result = subprocess.run(
        ["rclone", "mkdir", folder],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    print("Verified:", folder)

# ------------------------------------------------------------
# 5. TEST GOOGLE DRIVE ACCESS
# ------------------------------------------------------------

result = subprocess.run(
    ["rclone", "lsd", DRIVE_ROOT],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

print("\nGoogle Drive access successful.")

print("\n" + "=" * 65)
print("STAGE 3 COMPLETED SUCCESSFULLY")
print("=" * 65)

In [ ]:
!rclone config reconnect gdrive: --no-browser

In [ ]:
import subprocess

print("Checking Google Drive...")

result = subprocess.run(
    ["rclone", "lsd", "gdrive:"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("Google Drive authentication successful!")

In [ ]:
# Install rclone
import subprocess

print("Installing rclone...")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "rclone"],
    check=True
)

# Verify installation
result = subprocess.run(
    ["rclone", "version"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)
print("RCLONE INSTALLED SUCCESSFULLY")

In [ ]:
# ============================================================
# STAGE 3B: UPDATE GOOGLE DRIVE AUTHENTICATION
# ============================================================

import base64
import configparser
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Read existing configuration and fresh authorization token
old_config = secrets.get_secret("RCLONE_CONFIG_B64")
new_token = secrets.get_secret("RCLONE_NEW_TOKEN")

# Decode existing rclone configuration
config_text = base64.b64decode(old_config).decode("utf-8")

config = configparser.ConfigParser(interpolation=None)
config.read_string(config_text)

assert "gdrive" in config.sections(), "gdrive remote missing"

# Replace only the Google Drive token
config.set("gdrive", "token", new_token)

# Generate updated configuration
from io import StringIO

buffer = StringIO()
config.write(buffer)

updated_config = base64.b64encode(
    buffer.getvalue().encode("utf-8")
).decode("utf-8")

print("=" * 60)
print("UPDATED RCLONE CONFIGURATION GENERATED")
print("=" * 60)

print("\nCopy the following value into Kaggle Secret:")
print("\nRCLONE_CONFIG_B64:")
print(updated_config)

print("\nKeep this configuration private.")

In [ ]:
import base64
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

value = secrets.get_secret("RCLONE_CONFIG_B64").strip()

print("Secret accessible:", bool(value))
print("Length:", len(value))
print("Length divisible by 4:", len(value) % 4 == 0)

try:
    decoded = base64.b64decode(value + "=" * (-len(value) % 4))
    text = decoded.decode("utf-8")

    print("Base64 decoding: SUCCESS")
    print("Configuration is valid UTF-8:", True)
    print("Contains gdrive section:", "[gdrive]" in text)

except Exception as e:
    print("Base64 decoding: FAILED")
    print("Error type:", type(e).__name__)

In [1]:
import base64
import configparser
from io import StringIO
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Read fresh authorization JSON
token_json = secrets.get_secret("RCLONE_NEW_TOKEN")

# Build rclone configuration
config = configparser.ConfigParser(interpolation=None)

config["gdrive"] = {
    "type": "drive",
    "scope": "drive",
    "token": token_json,
}

# Encode configuration
buffer = StringIO()
config.write(buffer)

config_b64 = base64.b64encode(
    buffer.getvalue().encode("utf-8")
).decode("utf-8")

print("Configuration generated successfully.")
print("\nCopy the following value into Kaggle Secrets:")
print(config_b64)

Configuration generated successfully.

Copy the following value into Kaggle Secrets:
W2dkcml2ZV0KdHlwZSA9IGRyaXZlCnNjb3BlID0gZHJpdmUKdG9rZW4gPSBQYXN0ZSB0aGUgZm9sbG93aW5nIGludG8geW91ciByZW1vdGUgbWFjaGluZSAtLS0+IHsiYWNjZXNzX3Rva2VuIjoieWEyOS5hMEFlaGZUT25iQ1NuZHdIbXlYdGkwcU13bG9HS0lUQjdxQjRnZEk4ZVRMc3FvanRiZ2NDREh5TmNibzlJbFJGOWxnUTlLbExUUVAyZkktazlFaWxwNURSTDRQMGlweVlvMDlQRGJ3dW5QSTRXMlE2aTRsSnUxX21jblBWYnJHMDFxS0g1V0tpYjE3a1JZZndwVE1LSXZEZzRJMWdERTZMYmFsZG55ZFgzVUNEd0pHOVNFSkYwQVA5N1ZaZlBjaml5QzY3anhjcGdhQ2dZS0FlRVNBUk1TRlFIR1gyTWlSY3NjeVp1c0VLWHlIU2xVZ1FaMVd3MDIwNiIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJyZWZyZXNoX3Rva2VuIjoiMS8vMGdnVXhHWC0xWDFGbENnWUlBUkFBR0JBU1NRRi1MOUlyUUM3VlFBMFhrdjRmblR2UkRSVUZGZlNTRjhna2VSWlJDM0VCQlNJdVAzNVZNa1NqSTR2WFdxOWViRi1LYTdkR0FvclJpSWhtTG9wSWVjTjFac1dsR1VMSEw5NCIsImV4cGlyeSI6IjIwMjYtMTEtMjBUMjI6NDA6MjMuMjk0NjgxOSswNTozMCIsImV4cGlyZXNfaW4iOjM1OTl9Cgo=


In [3]:
import json
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("RCLONE_NEW_TOKEN")

data = json.loads(token)

assert data.get("access_token")
assert data.get("refresh_token")

print("OAuth JSON validated successfully.")

OAuth JSON validated successfully.
